Directory three of the project:

```
├── code
├── data
├── jmxs
├── model
├── pcaps
└── results
```

The working dir of the script is the code/ directory

The code will be used to extract data to instrument the model. The data has been obtained from the system executing the application. The perform N readings in a observation time interval T. 

    1. read data from the data dir/ - 
    
    ```
    ├── 10-38
    │   ├── containers_pre.json
    │   ├── containers_post.json
    │   ├── energy.csv
    │   ├── system_pre.json
    │   ├── system_post.json
    │   └── requests.jtl
    ├── 11-38
    │   ├── containers_pre.json
    │   ├── containers_post.json
    │   ├── energy.csv
    │   ├── system_pre.json
    │   ├── system_post.json
    │   └── requests.jtl
    ```
       
    The name of the directory contains two numbers. The former describes the ith reading while the second one the population of customers operating while reading performance and energy values.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from itertools import chain
import csv, json, glob, os

In [2]:
IPSPATH = "experiment_configuration_data/ips.csv"
CONVERSATIONSPATH = "experiment_configuration_data/conversations.csv"
ips = pd.read_csv(IPSPATH)
conversations = pd.read_csv(CONVERSATIONSPATH)

In [3]:
for i, x in ips.iterrows():
    conversations = conversations.replace(x['ip'], x['container'])

In [4]:
conversations = pd.read_csv("experiment_configuration_data/conversations_names.csv")

In [5]:
conversations

,source,destination
0,145.108.225.16,145.108.225.7
1,145.108.225.16,ui
2,ui,auth
3,auth,verification-code
4,ui,verification-code
5,ui,travel
6,travel,ticketinfo
7,ticketinfo,basic
8,basic,station
9,travel,route


In [6]:
conversations[(conversations == 'travel').any(axis=1)]

,source,destination
5,ui,travel
6,travel,ticketinfo
9,travel,route
13,travel,order
14,travel,seat
16,travel,train
22,food,travel
29,preserve,travel


In [7]:
DATA = f"../model/results-single"
DIRS = list(map(lambda x: f"{DATA}/{x}", os.listdir(DATA)))

In [8]:
DIRS75 = list(filter(lambda x: x.find("-75") != -1, DIRS))

In [9]:
def get_energy_stats(trials):
    time   = np.array([len(x['time']) for x in trials])
    energy = np.array(
        [np.trapz(x['power'], x['time']) for x in trials]
    )
    e = energy/time
    
    print(
            f"# Energy Per Visit(Joule/Visit):\n",
            f"## Mean:\t\t\t{energy.mean()}", 
            f"## Min-Max:\t\t\t[{energy.min()}, {energy.max()}]",
            f"## Var:\t\t\t\t{energy.var()}", 
            f"## Std:\t\t\t\t{energy.std()}", 
            '\n'
            f"# Average Response Time:\t{time.mean()}",
            f"# e (Joule/s):\t\t\t{e.mean()}, [{e.min()}, {e.max()}]",
            sep='\n'
         )

In [10]:
ENERGYFILES = [f"{x}/energy.csv" for x in DIRS]

In [11]:
energy_values = [
    pd.read_csv(x, names = ["time", "power"]) for x in ENERGYFILES
]

In [12]:
get_energy_stats(energy_values)

# Energy Per Visit(Joule/Visit):

## Mean:			742.3410799999999
## Min-Max:			[710.1889000000001, 807.83495]
## Var:				1016.1577052245999
## Std:				31.877228631494926

# Average Response Time:	13.6
# e (Joule/s):			54.60543380586081, [52.516392857142854, 55.63896071428571]


In [13]:
from helpers.stats import SystemStats
from helpers.stats import ContainerStats

In [14]:
def calculate_containers_utilization(DIRS):
    rows = []
    for x in DIRS:
        f1, f2 = glob.glob(f"{x}/containers_*.json")
        data = ContainerStats(f2, f1).data
        rows.append(
            {k: [v['cpu'], v['disk'], v['io']] for k,v in data.items()}
        )

    return pd.DataFrame.from_records(rows)

def calculate_system_utilization(DIRS):
    rows = []
    for x in DIRS:
        f1, f2 = glob.glob(f"{x}/system_*.json")
        data = SystemStats(f2, f1).data[0]
        rows.append(
            {'cpu': data['cpu0'], 'duration': data['duration']}
        )

    return pd.DataFrame.from_records(rows)

In [15]:
containers_stats = calculate_containers_utilization(DIRS75)
system_stats = calculate_system_utilization(DIRS75)
mean_duration = system_stats['duration'].mean()
mean_cpu = system_stats['cpu'].mean()
arrival_rate = 1/mean_duration

In [16]:
print(f"mean_cpu: {mean_cpu}", f"mean_duration: {mean_duration}", f"arrival_rate: {arrival_rate}")

mean_cpu: 46.1371 mean_duration: 14.20819113254547 arrival_rate: 0.0703819360727339


In [17]:
#containers_stats.drop('docker', axis=1).head(1).values.sum()

In [18]:
for k in containers_stats.keys():
    cpu_util = np.array([v[0] for v in containers_stats[k]])
    #disk_util = np.array([v[1] for v in containers_stats[k]])
    io_util = np.array([v[2] for v in containers_stats[k]])
    mean_cpu = cpu_util.mean()
    mean_io = io_util.mean()
    #mean_disk = disk_util.mean()
    print(f"{k:<50} {mean_cpu:<20} {((mean_cpu/100)/arrival_rate):<20} {mean_io}")

baseline-ts-consign-price-service-1                0.11177240944098657  0.01588083756682667  0.4
baseline-ts-price-service-1                        0.38930354406703244  0.05531299162681736  0.2
baseline-ts-contacts-service-1                     0.6713189663401817   0.09538228184664163  0.4
baseline-ts-order-service-1                        1.289217592962197    0.18317449972247105  0.1
baseline-ts-route-service-1                        1.4897847942377458   0.2116714710288982   0.2
baseline-ts-travel-service-1                       3.2848188931444207   0.4667133466959239   0.0
baseline-ts-user-service-1                         0.16118791545888736  0.02290188711096453  0.6
baseline-ts-config-service-1                       0.4372045481392325   0.062118857839803934 0.0
baseline-ts-ticketinfo-service-1                   1.8634332559498366   0.26476015863276803  0.0
baseline-ts-order-other-service-1                  0.14299201342917522  0.020316578572292306 0.0
baseline-ts-station-service-1 